# Lab 4 — Bayesian Decision Theory
**ENGS 3302 Pattern Recognition · Week 4** · Notes: *Week 4 — Bayesian Decision Theory*

| Part | What you do | How |
|---|---|---|
| **1. Demo** | The prior alone, the posterior, the boundary and the total error, loss and risk, three penguin species | All code is given. **Watch the lab video, run each cell, change values** |
| **2. Your turn** | Small tasks on the same ideas | Fill the `# TODO` cells. A check cell tells you if you are right |

This lab has **no grade**. The graded work this week is `Assignment_W04`.
In Colab: *File → Save a copy in Drive* first, then run the cells from top to bottom.

In [ ]:
# Run this cell first. It loads the libraries, a helper that checks your answers, and the penguin data.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.model_selection import train_test_split

def check(label, got, expected, tol=1e-2):
    """Prints a green tick if `got` is close to `expected`, otherwise a red cross."""
    try:
        if isinstance(expected, str):
            ok = str(got) == expected
        else:
            ok = got is not None and np.allclose(np.asarray(got, dtype=float),
                                                 np.asarray(expected, dtype=float), atol=tol)
    except Exception:
        ok = False
    print(f"✅ {label}" if ok else f"❌ {label}: expected {expected}, got {got}")

# Palmer Penguins (Horst, Hill & Gorman 2020, CC0): 344 penguins of 3 species, measured in mm and g.
# Uses penguins.csv from the Moodle zip if it is next to the notebook, otherwise downloads it.
PENGUIN_FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
def load_penguins():
    """Returns the cleaned table: 342 penguins with all four measurements."""
    try:
        df = pd.read_csv("penguins.csv")
    except FileNotFoundError:
        df = pd.read_csv("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv")
    return df.dropna(subset=PENGUIN_FEATURES).reset_index(drop=True)   # 2 penguins have no measurements

peng = load_penguins()
print("Ready.", len(peng), "penguins")

---
## Part 1 — Demo *(follow the lab video)*
### 1.1 A factory of fish
We simulate 100 000 fish with the models of the Notes: 60% sea bass with lightness $N(7,1)$, 40% salmon with lightness $N(4,1)$.
First rule: no camera, always say *sea bass* (Worked example 4.1).

In [ ]:
rng = np.random.default_rng(0)                 # fixed seed: everyone gets the same fish
n = 100000
is_bass = rng.random(n) < 0.6                  # True = sea bass
x = np.where(is_bass, rng.normal(7, 1, n), rng.normal(4, 1, n))   # lightness of each fish
print("sea bass:", is_bass.sum(), " salmon:", (~is_bass).sum())
print("error of 'always sea bass':", np.mean(~is_bass))           # about 0.4

### 1.2 The posterior of one fish
Bayes' rule in three steps for $x=5.4$ (Worked example 4.2). Then the same fish with other priors.

In [ ]:
def posterior(x, p_bass=0.6):
    s_bass   = norm.pdf(x, 7, 1) * p_bass          # likelihood x prior
    s_salmon = norm.pdf(x, 4, 1) * (1 - p_bass)
    return s_bass / (s_bass + s_salmon)            # divide by the evidence

print("likelihoods at 5.4:", round(norm.pdf(5.4, 7, 1), 3), round(norm.pdf(5.4, 4, 1), 3))
for p in (0.2, 0.5, 0.6, 0.8):
    print(f"P(sea bass) = {p}:  P(sea bass | 5.4) = {posterior(5.4, p):.3f}")

### 1.3 The boundary and the total error
Rule: decide sea bass when $x>t$. The Bayes boundary is the first $t$ where the sea bass score is larger. We count the mistakes on the 100 000 simulated fish for several thresholds $t$,
and compare with the two areas of the Notes (Figure 2): $0.6\cdot P(x<t\mid\text{sea bass}) + 0.4\cdot P(x>t\mid\text{salmon})$.

In [ ]:
t_grid = np.linspace(1, 10, 9001)
g1 = norm.pdf(t_grid, 7, 1) * 0.6
g2 = norm.pdf(t_grid, 4, 1) * 0.4
x_star = t_grid[np.argmax(g1 > g2)]    # first t where sea bass wins
print("Bayes boundary x* =", round(x_star, 3))      # 5.365

for t in (5.0, x_star, 5.5, 6.0):
    counted = np.mean((x > t) != is_bass)            # the decision differs from the truth
    areas   = 0.6 * norm.cdf(t, 7, 1) + 0.4 * (1 - norm.cdf(t, 4, 1))
    print(f"t = {t:.3f}:  counted {counted:.4f}   two areas {areas:.4f}")

### 1.4 Loss and risk
Calling a sea bass "salmon" costs 2; calling a salmon "sea bass" costs 1 (the loss table of §4.5). For one fish we compute the two risks.
Then we compare the cost of the two boundaries on all the simulated fish (Worked example 4.6).

In [ ]:
loss = np.array([[0, 1],     # alpha_1: decide sea bass   (true: sea bass, salmon)
                 [2, 0]])    # alpha_2: decide salmon
p1 = posterior(5.2)
risk = loss @ np.array([p1, 1 - p1])
print("posteriors at 5.2:", round(p1, 3), round(1 - p1, 3), "  risks:", risk.round(3), "  action:", np.argmin(risk))

def cost_of(t):
    say_bass = x > t
    wrong_bass_called_salmon = np.sum(is_bass & ~say_bass)
    wrong_salmon_called_bass = np.sum(~is_bass & say_bass)
    mistakes = wrong_bass_called_salmon + wrong_salmon_called_bass
    return mistakes / n, (2 * wrong_bass_called_salmon + 1 * wrong_salmon_called_bass) / n

for name, t in (("minimum error", 5.365), ("minimum risk ", 5.134)):
    e, c = cost_of(t)
    print(f"{name} t = {t}:  mistakes per fish {e:.4f}   cost per fish {c:.4f}")

### 1.5 Three penguin species
A Bayes classifier on real data: one normal curve per species for the flipper length, fitted on the **training** penguins,
and the share of each species in the training data as its prior. We test it on the penguins kept aside (Week 2: 70/30, `random_state=0`).

In [ ]:
train, test = train_test_split(peng, test_size=0.3, random_state=0, stratify=peng["species"])
species = ["Adelie", "Chinstrap", "Gentoo"]
params = {}
for sp in species:
    v = train.loc[train["species"] == sp, "flipper_length_mm"]
    params[sp] = (v.mean(), v.std(ddof=0), len(v) / len(train))     # mean, sd, prior
    print(f"{sp:10s} mean {params[sp][0]:.1f}  sd {params[sp][1]:.2f}  prior {params[sp][2]:.3f}")

def classify(x, use_priors=True):
    scores = [norm.pdf(x, m, s) * (p if use_priors else 1/3) for m, s, p in params.values()]
    return species[int(np.argmax(scores))]

pred = [classify(v) for v in test["flipper_length_mm"]]
print("test accuracy:", np.mean(np.array(pred) == test["species"].to_numpy()).round(3), "of", len(test), "penguins")
print("195 mm ->", classify(195), "   with equal priors ->", classify(195, use_priors=False))

---
## Part 2 — Your turn
### Task 1 — Posteriors for a rare class
Use `posterior()` from 1.2. On a day with only **30%** sea bass, compute $P(\text{sea bass}\mid x)$ for $x=6.0$, and store the decision (`"sea bass"` or `"salmon"`).

In [ ]:
p6 = None         # TODO
decision6 = None  # TODO

In [ ]:
check("P(sea bass | 6.0), 30% sea bass", p6, 0.658, tol=0.002)
check("decision at 6.0", decision6, "sea bass")

### Task 2 — Where is the boundary when sea bass are 80% of the catch?
Repeat the boundary search of 1.3 with the prior 0.8. Then compute the total error at that boundary with the two areas.

In [ ]:
x_star_80 = None   # TODO
error_80 = None    # TODO

In [ ]:
check("boundary with prior 0.8", x_star_80, 5.038, tol=0.002)
check("total error with prior 0.8", error_80, 0.0498, tol=0.0005)

### Task 3 — A more expensive mistake
Now calling a **salmon** "sea bass" costs 3 and the other mistake costs 1. Write the loss table, compute the two risks at $x=5.6$ (prior 0.6),
and store the action: 0 = decide sea bass, 1 = decide salmon.

In [ ]:
loss3 = None    # TODO: 2 x 2 array, rows = actions, columns = true class
risk3 = None    # TODO
action3 = None  # TODO

In [ ]:
check("risks at 5.6", risk3, [3 * (1 - posterior(5.6)), posterior(5.6)], tol=1e-3)
check("action at 5.6", action3, 1)

### Task 4 — One penguin, two kinds of prior
Use `classify()` from 1.5. Classify a penguin with flippers of **200 mm** with the priors and with equal priors. Do the two answers agree? (This classifier is fitted on the training penguins only, so its numbers differ a little from Worked example 4.7.)

In [ ]:
with_priors = None   # TODO
equal_priors = None  # TODO

In [ ]:
check("200 mm with priors", with_priors, "Adelie")
check("200 mm with equal priors", equal_priors, "Chinstrap")

### Reflect (2–3 sentences)
In 1.3 the threshold 5.5 (halfway between the centres) makes only a few more mistakes than the Bayes boundary 5.365. When would the difference be much larger?
(Hint: try `posterior` and the boundary search with a prior of 0.95.)

*Your answer:*